In [1]:
import os
import random
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from PIL import Image, ImageOps, ImageEnhance

# Seeds
random.seed(21)
torch.manual_seed(21)

# Paths
dataset_path = "/Users/justinhoffman/HTR"
images_path = os.path.join(dataset_path, "self_lines")
labels_path = os.path.join(dataset_path, "lines.txt")

# Parse labels
samples = []
with open(labels_path, "r") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        parts = line.split()
        filename = parts[0] + ".png"
        text = " ".join(parts[9:]).replace("|", " ")
        image_path = os.path.join(images_path, filename)
        if os.path.exists(image_path):
            samples.append({"image_path": image_path, "text": text})

# Split
random.shuffle(samples)
train_size = int(0.70 * len(samples))
val_size   = int(0.15 * len(samples))
train_samples = samples[:train_size]
val_samples   = samples[train_size:train_size + val_size]
test_samples  = samples[train_size + val_size:]

print(f"Train: {len(train_samples)} | Val: {len(val_samples)} | Test: {len(test_samples)}")

Train: 235 | Val: 50 | Test: 51


In [2]:
class HandwritingDataset(Dataset):
    def __init__(self, samples, processor, augment=False, max_target_length=128):
        self.samples = samples
        self.processor = processor
        self.augment = augment
        self.max_target_length = max_target_length

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        image = Image.open(sample["image_path"]).convert("RGB")
        image = ImageOps.grayscale(image)
        image = ImageOps.autocontrast(image)

        if self.augment:
            angle = random.uniform(-3, 3)
            image = image.rotate(angle, fillcolor=255)
            enhancer = ImageEnhance.Brightness(image)
            image = enhancer.enhance(random.uniform(0.8, 1.2))
            enhancer = ImageEnhance.Contrast(image)
            image = enhancer.enhance(random.uniform(0.8, 1.2))

        image = image.convert("RGB")
        pixel_values = self.processor(images=image, return_tensors="pt").pixel_values.squeeze()

        labels = self.processor.tokenizer(
            sample["text"],
            padding="max_length",
            max_length=self.max_target_length,
            truncation=True,
            return_tensors="pt"
        ).input_ids.squeeze()

        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        return {"pixel_values": pixel_values, "labels": labels}

processor = TrOCRProcessor.from_pretrained("microsoft/trocr-small-handwritten")

train_dataset = HandwritingDataset(train_samples, processor, augment=True)
val_dataset   = HandwritingDataset(val_samples,   processor, augment=False)
test_dataset  = HandwritingDataset(test_samples,  processor, augment=False)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=4, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=4, shuffle=False)

print("DataLoaders ready")

DataLoaders ready


In [5]:
from torch.optim import AdamW
from jiwer import cer, wer

def evaluate(model, processor, test_samples, device):
    model.eval()
    predictions = []
    ground_truths = []

    for sample in test_samples:
        image = Image.open(sample["image_path"]).convert("RGB")
        image = ImageOps.grayscale(image)
        image = ImageOps.autocontrast(image)
        image = image.convert("RGB")

        pixel_values = processor(images=image, return_tensors="pt").pixel_values.to(device)

        with torch.no_grad():
            generated_ids = model.generate(pixel_values, max_new_tokens=64)

        predicted_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
        predictions.append(predicted_text)
        ground_truths.append(sample["text"])

    return cer(ground_truths, predictions), wer(ground_truths, predictions)


def train(learning_rate, num_epochs, augment=True, label="experiment"):
    if torch.backends.mps.is_available():
        device = torch.device("mps")
    elif torch.cuda.is_available():
        device = torch.device("cuda")
    else:
        device = torch.device("cpu")

    print(f"\n{'='*50}")
    print(f"Experiment: {label}")
    print(f"LR: {learning_rate} | Epochs: {num_epochs} | Augment: {augment}")
    print(f"Device: {device}")
    print(f"{'='*50}")

    # Reload fresh model for each experiment
    model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-small-handwritten")
    model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
    model.config.pad_token_id = processor.tokenizer.pad_token_id
    model.config.eos_token_id = processor.tokenizer.sep_token_id
    model.to(device)

    optimizer = AdamW(model.parameters(), lr=learning_rate)

    # Rebuild DataLoaders with augment flag
    t_dataset = HandwritingDataset(train_samples, processor, augment=augment)
    t_loader  = DataLoader(t_dataset, batch_size=4, shuffle=True)

    history = {
        "train_loss": [],
        "val_cer": [],
        "val_wer": []
    }

    print("Starting training...")
    for epoch in range(num_epochs):
        # ── Training ─────────────────────────────────────────────
        model.train()
        total_loss = 0

        for batch in t_loader:
            pixel_values = batch["pixel_values"].to(device)
            labels       = batch["labels"].to(device)

            outputs = model(pixel_values=pixel_values, labels=labels)
            loss    = outputs.loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(t_loader)

        # ── Validation ───────────────────────────────────────────
        val_cer, val_wer = evaluate(model, processor, val_samples, device)

        history["train_loss"].append(avg_loss)
        history["val_cer"].append(val_cer)
        history["val_wer"].append(val_wer)

        print(f"Epoch {epoch+1}/{num_epochs} | Loss: {avg_loss:.4f} | Val CER: {val_cer:.2%} | Val WER: {val_wer:.2%}")

    # ── Final test evaluation ────────────────────────────────────
    test_cer, test_wer = evaluate(model, processor, test_samples, device)
    print(f"\nFinal Test CER: {test_cer:.2%} | Test WER: {test_wer:.2%}")

    return model, history, test_cer, test_wer

In [6]:
model_1, history_1, cer_1, wer_1 = train(
    learning_rate=5e-5,
    num_epochs=5,
    augment=True,
    label="lr=5e-5, epochs=5, augment=True"
)


Experiment: lr=5e-5, epochs=5, augment=True
LR: 5e-05 | Epochs: 5 | Augment: True
Device: mps


Loading weights:   0%|          | 0/360 [00:00<?, ?it/s]

VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-small-handwritten
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.bias   | MISSING | 
encoder.pooler.dense.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Starting training...


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/transformers/generation/utils.py:1569: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Epoch 1/5 | Loss: 18.2792 | Val CER: 181.58% | Val WER: 208.88%


KeyboardInterrupt: 